# Install and load model

In [1]:
!pip install llama-cpp-python \
  --extra-index-url https://abetlen.github.io/llama-cpp-python/whl/cu121 \
  huggingface_hub \
  python-docx

Looking in indexes: https://pypi.org/simple, https://abetlen.github.io/llama-cpp-python/whl/cu121


In [ ]:
import json
from llama_cpp import Llama
from huggingface_hub import hf_hub_download
model_name = "bartowski/Qwen2.5-14B-Instruct-GGUF"
model_file = "Qwen2.5-14B-Instruct-Q4_K_M.gguf"

print(f"Đang kiểm tra/tải model {model_file}...")
model_path = hf_hub_download(repo_id=model_name, filename=model_file)
print(f"Model path: {model_path}")

llm = Llama(
    model_path=model_path,
    n_gpu_layers=-1,
    n_ctx=16384,
    verbose=False
)

Đang kiểm tra/tải model Qwen2.5-14B-Instruct-Q4_K_M.gguf...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Model path: /root/.cache/huggingface/hub/models--bartowski--Qwen2.5-14B-Instruct-GGUF/snapshots/05244aa5d871c661c80082a15d3bce44714d068d/Qwen2.5-14B-Instruct-Q4_K_M.gguf


llama_context: n_ctx_per_seq (16384) < n_ctx_train (32768) -- the full capacity of the model will not be utilized


# Prompts

In [3]:
# 2. ĐỊNH NGHĨA FULL SCHEMA (Bắt buộc phải chi tiết từng trường)
# Đây là bản đồ hướng dẫn model phải điền vào đâu
complex_schema = {
    "type": "object",
    "properties": {
        "course_id": {"type": "string", "description": "Unique identifier like DATA101"},
        "title": {"type": "string", "description": "Official name of the course"},
        "difficulty_level": {"type": "string", "enum": ["Beginner", "Intermediate", "Advanced", "Expert"]},
        "duration": {"type": "string", "description": "Duration of the course"},
        "skill_outcomes": {
            "type": "array",
            "items": {
                "type": "object",
                "properties": {
                    "skill_name": {"type": "string",
                                   "description": "Standardized professional skill name (closest to ESCO terms)."},
                    "category": {"type": "string", "enum": ["Hard Skill", "Soft Skill", "Tool/Software"]},
                    "outcome_description": {"type": "string", "description": "The specific learning outcome context from the document." }
                },
                "required": ["skill_name", "category", "outcome_description"]
            }
        },
        "entry_requirements": {
             "type": "object",
             "properties": {
                 "prerequisite_courses": {"type": "array", "items": {"type": "string"}},
                 "minimum_entry_skills": {
                     "type": "array",
                     "items": {
                         "type": "object",
                         "properties": {
                             "skill_name": {"type": "string"},
                             "minimum_proficiency_level": {"type": "integer"}
                         },
                         "required": ["skill_name", "minimum_proficiency_level"]
                     }
                 }
             },
             "required": ["prerequisite_courses", "minimum_entry_skills"]
        },
        "pedagogy_format": {
            "type": "object",
            "properties": {
                "has_project": {"type": "boolean"}
            },
            "required": ["has_project"]
        }
    },
    "required": ["course_id", "title", "difficulty_level", "duration", "skill_outcomes", "pedagogy_format"]
}

# Infer

In [ ]:
import json
import os
from docx import Document

# ==========================================
# 1. HÀM ĐỌC DOCX (TEXT + TABLES)
# ==========================================
def read_docx_optimized(file_path):
    """
    Đọc file docx, lấy cả văn bản và nội dung trong bảng.
    Gộp lại thành string sạch để tiết kiệm token.
    """
    if not os.path.exists(file_path):
        raise FileNotFoundError(f"Không tìm thấy file: {file_path}")

    doc = Document(file_path)
    full_text = []

    # 1. Đọc các đoạn văn (Paragraphs)
    print("...Đang đọc các đoạn văn...")
    for para in doc.paragraphs:
        text = para.text.strip()
        if text: # Chỉ lấy dòng có chữ
            full_text.append(text)

    # 2. Đọc các bảng (Tables) - Rất quan trọng với đề cương môn học
    print("...Đang đọc các bảng biểu...")
    for table in doc.tables:
        for row in table.rows:
            # Gộp các cell trong 1 hàng bằng dấu " | " để giả lập định dạng Markdown table
            row_text = " | ".join(cell.text.strip() for cell in row.cells)
            if row_text.strip():
                full_text.append(f"| {row_text} |")

    # Nối lại thành 1 chuỗi lớn
    result_text = "\n".join(full_text)
    
    # LOG: Độ dài text và ước tính tokens
    char_count = len(result_text)
    estimated_tokens = char_count // 4  # Ước tính thô: ~4 chars = 1 token
    print(f"✓ Đọc xong file: {char_count:,} ký tự (~{estimated_tokens:,} tokens ước tính)")
    
    return result_text


# Extract

In [6]:
def extract_course_data(file_path, llm_instance):
    """Gửi nội dung file vào LLM để trích xuất JSON."""
    context_text = read_docx_optimized(file_path)

    if not context_text:
        return None

    # Prompt System: Hướng dẫn đóng vai trò trích xuất dữ liệu
    messages = [
        {
            "role": "system",
            "content": (
                """
            You are an expert Job Analyst and HR Specialist familiar with the ESCO (European Skills, Competences, Qualifications and Occupations) framework.

            Your task is to analyze course documents and extract structured data.
            CRITICAL INSTRUCTION FOR SKILLS:
            1. Do NOT simply copy the learning objectives from the text.
            2. You must GENERALIZE and TRANSLATE specific academic tasks into standard professional skills found in ESCO.
            3. Avoid granular steps (e.g., instead of "solving linear equations", use "Linear algebra" or "Mathematics").
            4. Instead of "writing for-loops in Python", use "Python (Computer Programming)".
            5. Keep skill names concise (1-4 words).
            """
            )
        },
        {"role": "user", "content": context_text}
    ]

    try:
        response = llm_instance.create_chat_completion(
            messages=messages,
            response_format={
                "type": "json_object",
                "schema": complex_schema
            },
            temperature=0.1,
            max_tokens=16000
        )

        content = response["choices"][0]["message"]["content"]
        return json.loads(content)

    except Exception as e:
        print(f"Error extracting AI data for {file_path}: {e}")
        return None

# ==========================================
# 4. HÀM DUYỆT THƯ MỤC VÀ XỬ LÝ BATCH
# ==========================================
def process_course_folder(input_root_dir, output_root_dir, llm_instance):
    """
    input_root_dir: Folder chứa các folder con và file docx đầu vào.
    output_root_dir: Folder sẽ chứa các file json đầu ra (giữ nguyên cấu trúc).
    """
    print(f"Bắt đầu quét thư mục: {input_root_dir}")

    for root, dirs, files in os.walk(input_root_dir):
        for filename in files:
            # 1. Chỉ xử lý file .docx và bỏ qua file tạm (bắt đầu bằng ~$)
            if filename.endswith(".docx") and not filename.startswith("~$"):

                # Đường dẫn file nguồn
                input_file_path = os.path.join(root, filename)
                print(f"\nDang xu ly: {input_file_path}")

                # 2. Tạo đường dẫn file đích (Output mirroring)
                # Tính đường dẫn tương đối (ví dụ: SubFolder/Course1.docx)
                relative_path = os.path.relpath(root, input_root_dir)

                # Tạo folder đích tương ứng nếu chưa có
                target_dir = os.path.join(output_root_dir, relative_path)
                os.makedirs(target_dir, exist_ok=True)

                # Tên file json output
                output_filename = os.path.splitext(filename)[0] + ".json"
                output_file_path = os.path.join(target_dir, output_filename)

                # 3. Kiểm tra nếu file JSON đã tồn tại thì bỏ qua (Resume capability)
                if os.path.exists(output_file_path):
                    print(f"--> File JSON da ton tai, bo qua: {output_filename}")
                    continue

                # 4. Gọi hàm AI extract
                extracted_data = extract_course_data(input_file_path, llm_instance)

                # 5. Lưu file JSON
                if extracted_data:
                    with open(output_file_path, 'w', encoding='utf-8') as f:
                        json.dump(extracted_data, f, indent=2, ensure_ascii=False)
                    print(f"--> Da luu xong: {output_file_path}")
                else:
                    print(f"--> Khong trich xuat duoc du lieu: {filename}")

In [ ]:
if __name__ == "__main__":
    # Cấu hình đường dẫn
    INPUT_DIR = "/content/drive/MyDrive/course recommendation/Viện Ngân hàng Tài chính_2024"  # Thay bằng đường dẫn thực tế của bạn
    OUTPUT_DIR = "/content/drive/MyDrive/course recommendation/Data_Courses_Json" # Folder output

    process_course_folder(INPUT_DIR, OUTPUT_DIR, llm)

    print("Vui lòng khởi tạo biến 'llm' và uncomment dòng process_course_folder để chạy.")

Bắt đầu quét thư mục: /content/drive/MyDrive/course recommendation/Viện Ngân hàng Tài chính_2024

Dang xu ly: /content/drive/MyDrive/course recommendation/Viện Ngân hàng Tài chính_2024/Khoa Quản trị kinh doanh_2024/Quản trị công ty_QTKD1134.docx
...Đang đọc các đoạn văn...
...Đang đọc các bảng biểu...
